## Deploy — Streamlit Demo

### By:
Data Science Team

### Date:
2026-02-25

### Description:
This notebook:
1. **Validates** the prediction logic in-notebook (sanity check before deploy)  
2. **Writes** the Streamlit application to `app/streamlit_app.py`  
3. **Provides** instructions to run the demo locally  

The Streamlit app loads the serialised model from `data/06_models/` and lets a
user fill in customer features to get a real-time churn probability prediction.

## 📚 Import libraries

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import joblib
import pandas as pd

warnings.filterwarnings("ignore")
print("Python:", sys.version)

## ✅ Sanity Check — In-notebook Prediction Test

In [ ]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"
MODELS_DIR = DATA_DIR / "06_models"
CHURN_THRESHOLD = 0.5

model = joblib.load(MODELS_DIR / "best_model.pkl")
preprocessor = joblib.load(MODELS_DIR / "preprocessor.pkl")

with open(MODELS_DIR / "feature_names.json") as f:
    feature_names = json.load(f)
with open(MODELS_DIR / "model_metadata.json") as f:
    metadata = json.load(f)

print(f"✅ Loaded model: {metadata['model_name']}")
print(f"   Test AUC-ROC : {metadata['test_metrics']['auc_roc']}")

# Simulate one customer (raw, before preprocessing)
sample_customer = pd.DataFrame(
    [
        {
            "SeniorCitizen": 0,
            "Partner": "Yes",
            "Dependents": "No",
            "tenure": 12,
            "MultipleLines": "No",
            "InternetService": "Fiber optic",
            "OnlineSecurity": "No",
            "OnlineBackup": "No",
            "DeviceProtection": "No",
            "TechSupport": "No",
            "StreamingTV": "Yes",
            "StreamingMovies": "Yes",
            "Contract": "Month-to-month",
            "PaperlessBilling": "Yes",
            "PaymentMethod": "Electronic check",
            "MonthlyCharges": 79.85,
            "TotalCharges": 958.20,
        }
    ]
)

transformed = preprocessor.transform(sample_customer)
proba = model.predict_proba(transformed)[0, 1]
prediction = "CHURN ⚠️" if proba >= CHURN_THRESHOLD else "NO CHURN ✅"

print("\n--- Sample Prediction ---")
print(sample_customer.T.to_string())
print(f"\nChurn probability : {proba:.2%}")
print(f"Prediction        : {prediction}")

## 🖊️ Write Streamlit App to `app/streamlit_app.py`

In [ ]:
APP_DIR = Path.cwd().resolve().parents[1] / "app"
APP_DIR.mkdir(parents=True, exist_ok=True)

STREAMLIT_CODE = '''"""
Streamlit Demo — Customer Churn Predictor
Run: streamlit run app/streamlit_app.py
"""
import json
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR   = Path(__file__).resolve().parents[1]
MODELS_DIR = BASE_DIR / "data" / "06_models"

# ── Load artefacts ─────────────────────────────────────────────────────────────
@st.cache_resource
def load_artefacts():
    model        = joblib.load(MODELS_DIR / "best_model.pkl")
    preprocessor = joblib.load(MODELS_DIR / "preprocessor.pkl")
    with open(MODELS_DIR / "model_metadata.json") as f:
        meta = json.load(f)
    return model, preprocessor, meta

model, preprocessor, meta = load_artefacts()

# ── UI ─────────────────────────────────────────────────────────────────────────
st.set_page_config(page_title="Churn Predictor", page_icon="📡", layout="centered")
st.title("📡 Customer Churn Predictor")
st.caption(f"Model: **{meta[\'model_name\']}**  |  Test AUC-ROC: **{meta[\'test_metrics\'][\'auc_roc\']}**")

st.header("Customer Features")

col1, col2, col3 = st.columns(3)

with col1:
    senior          = st.selectbox("Senior Citizen", [0, 1], format_func=lambda x: "Yes" if x else "No")
    partner         = st.selectbox("Partner", ["Yes", "No"])
    dependents      = st.selectbox("Dependents", ["Yes", "No"])
    tenure          = st.slider("Tenure (months)", 0, 72, 12)
    multiple_lines  = st.selectbox("Multiple Lines", ["No", "Yes", "No phone service"])

with col2:
    internet_svc    = st.selectbox("Internet Service", ["DSL", "Fiber optic", "No"])
    online_sec      = st.selectbox("Online Security", ["Yes", "No", "No internet service"])
    online_bkp      = st.selectbox("Online Backup",   ["Yes", "No", "No internet service"])
    device_prot     = st.selectbox("Device Protection",["Yes", "No", "No internet service"])
    tech_support    = st.selectbox("Tech Support",    ["Yes", "No", "No internet service"])

with col3:
    streaming_tv    = st.selectbox("Streaming TV",    ["Yes", "No", "No internet service"])
    streaming_mv    = st.selectbox("Streaming Movies",["Yes", "No", "No internet service"])
    contract        = st.selectbox("Contract", ["Month-to-month", "One year", "Two year"])
    paperless       = st.selectbox("Paperless Billing", ["Yes", "No"])
    payment         = st.selectbox("Payment Method", [
        "Electronic check", "Mailed check",
        "Bank transfer (automatic)", "Credit card (automatic)"
    ])

monthly_charges = st.number_input("Monthly Charges ($)", 0.0, 200.0, 79.85, step=0.5)
total_charges   = st.number_input("Total Charges ($)",   0.0, 10000.0, float(tenure * monthly_charges), step=1.0)

# ── Predict ────────────────────────────────────────────────────────────────────
if st.button("🔮 Predict Churn", type="primary", use_container_width=True):
    input_df = pd.DataFrame([{
        "SeniorCitizen":    senior,
        "Partner":          partner,
        "Dependents":       dependents,
        "tenure":           tenure,
        "MultipleLines":    multiple_lines,
        "InternetService":  internet_svc,
        "OnlineSecurity":   online_sec,
        "OnlineBackup":     online_bkp,
        "DeviceProtection": device_prot,
        "TechSupport":      tech_support,
        "StreamingTV":      streaming_tv,
        "StreamingMovies":  streaming_mv,
        "Contract":         contract,
        "PaperlessBilling": paperless,
        "PaymentMethod":    payment,
        "MonthlyCharges":   monthly_charges,
        "TotalCharges":     total_charges,
    }])

    transformed = preprocessor.transform(input_df)
    proba       = model.predict_proba(transformed)[0, 1]

    st.divider()
    if proba >= CHURN_THRESHOLD:
        st.error(f"⚠️ **HIGH CHURN RISK** — Probability: **{proba:.1%}**")
    else:
        st.success(f"✅ **LOW CHURN RISK** — Probability: **{proba:.1%}**")

    st.progress(float(proba), text=f"Churn probability: {proba:.1%}")
'''

app_file = APP_DIR / "streamlit_app.py"
app_file.write_text(STREAMLIT_CODE)

print(f"✅ Streamlit app written to: {app_file}")

In [ ]:
print("=" * 60)
print("  HOW TO RUN THE STREAMLIT DEMO")
print("=" * 60)
print("""
1. Make sure all notebooks 1-5 have been executed (data must
   exist in data/05_model_input/ and data/06_models/).

2. Install Streamlit if needed:
      pip install streamlit

3. From the project root, run:
      streamlit run app/streamlit_app.py

4. The browser will open automatically at http://localhost:8501
""")

## 📊 Analysis of Results and Conclusions

- The sanity check in cell 5 verifies the **full prediction pipeline** (raw input → preprocessor → model → probability) end-to-end before deployment.
- The Streamlit app written to `app/streamlit_app.py` loads artefacts from `data/06_models/` and exposes a form UI.
- **No model is retrained** at serving time; all artefacts come from notebooks 4 & 5.

## 💡 Proposals and Ideas

1. **FastAPI backend**: decouple the model from the UI using the inference pipeline (notebook 8 / `src/pipelines/inference_pipeline/`).
2. **Docker container**: package the app and its dependencies for reproducible deployment.
3. **CI/CD**: trigger model retraining when new data arrives.

## 📖 References
- https://joserzapata.github.io/post/ciencia-datos-proyecto-python/7-deploy/
- Streamlit docs: https://docs.streamlit.io